In [2]:
!pip install sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

import google.generativeai as genai

from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
GOOGLE_API_KEY = userdata.get(
    "Research++"
)

genai.configure(
    api_key=GOOGLE_API_KEY
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print(
    "Gemini configured successfully."
)

Gemini configured successfully.


In [5]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving papers.json to papers.json


Saving researchforge_faiss.index to researchforge_faiss.index


In [6]:
df = pd.read_json(
    "papers.json"
)

faiss_index = faiss.read_index(
    "researchforge_faiss.index"
)

print(
    f"Loaded {len(df)} papers."
)

Loaded 100 papers.


In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model ready."
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [8]:
def retrieve_papers(
    user_query,
    top_k=10
):

    query_embedding = embedding_model.encode(
        [user_query]
    )

    distances, indices = faiss_index.search(
        np.array(
            query_embedding
        ).astype("float32"),
        k=top_k
    )

    retrieved_papers = []

    for idx in indices[0]:

        retrieved_papers.append({

            "title":
            df.iloc[idx]["title"],

            "abstract":
            df.iloc[idx]["abstract"],

            "published":
            df.iloc[idx]["published"]

        })

    return retrieved_papers

In [9]:
def detect_contradictions(
    user_query,
    retrieved_papers
):

    evidence_context = ""

    for paper in retrieved_papers:

        evidence_context += f"""

Title:
{paper['title']}

Abstract:
{paper['abstract']}

"""

    prompt = f"""

You are an expert scientific contradiction analyst.

User Question:

{user_query}

Retrieved Evidence:

{evidence_context}

Task:

Identify conflicting scientific claims.

Return:

1. Claim A

2. Opposing Claim B

3. Source evidence supporting each side

4. Possible causes of disagreement

5. Dataset differences

6. Architecture differences

7. Evaluation differences

8. Final evidence-grounded interpretation

Only use retrieved evidence.

Do not invent unsupported claims.

"""

    response = llm.generate_content(
        prompt
    )

    return response.text

In [10]:
query = """
Do transformer-based multimodal
deepfake detectors outperform CNN-based methods?
"""

papers = retrieve_papers(
    query
)

analysis = detect_contradictions(
    query,
    papers
)

print(
    analysis
)

The retrieved evidence presents a nuanced picture regarding the performance of transformer-based multimodal deepfake detectors versus CNN-based methods. While some papers highlight the cutting-edge performance of Transformers and Large Multimodal Models (LMMs), others emphasize the fundamental limitations of even state-of-the-art detection models, including those based on Transformers, when faced with novel or challenging deepfakes.

**1. Claim A:** Transformer-based and Large Multimodal Models (LMMs) achieve state-of-the-art performance and demonstrate superior generalization capabilities in deepfake detection, outperforming prior methods on various benchmarks.

**2. Opposing Claim B:** Despite advancements, state-of-the-art deepfake detection models, including Transformer-based ones, exhibit significant limitations in generalizing to unseen deepfake generation techniques or struggling with fine-grained, challenging deepfake attacks, leading to poor performance in real-world scenarios